# 加密货币 K 线分析

从 PostgreSQL 读取 OHLCV 数据，绘制交互式 K 线图 + 技术指标。

**数据来源**：数据库 `ohlcv` 表  
**图表库**：Plotly（可缩放、悬停查看详情）

## ⚙️ 配置 — 在这里修改币种和日期区间

In [ ]:
# ── 在这里修改参数 ──────────────────────────────────────────────
SYMBOL     = "BTC/USDT"   # 数据库中的币种标识，例如 "ETH/USDT"
START_DATE = "2024-01-01" # 开始日期 YYYY-MM-DD
END_DATE   = "2024-12-31" # 结束日期 YYYY-MM-DD（不含）
DB_TABLE   = "ohlcv"      # 数据库表名
# ────────────────────────────────────────────────────────────────

## 1. 导入 & 数据加载

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.db import read_ohlcv

# 从数据库读取
df = read_ohlcv(SYMBOL, start=START_DATE, end=END_DATE, table=DB_TABLE)

print(f"Symbol : {SYMBOL}")
print(f"Period : {START_DATE} ~ {END_DATE}")
print(f"Rows   : {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 2. 数据质检

In [ ]:
print("=== 缺失值 ===")
print(df.isnull().sum())

print("\n=== 基本统计 ===")
print(df[["open", "high", "low", "close", "volume"]].describe().round(4))

# 检查 high >= low 和 high >= close
anomalies = df[(df["high"] < df["low"]) | (df["high"] < df["close"]) | (df["low"] > df["close"])]
print(f"\n异常 OHLC 行数: {len(anomalies)}")
if not anomalies.empty:
    print(anomalies)

## 3. 技术指标计算

In [ ]:
d = df.copy()

# 均线
d["ema9"]  = d["close"].ewm(span=9,  adjust=False).mean()
d["ema21"] = d["close"].ewm(span=21, adjust=False).mean()
d["ema55"] = d["close"].ewm(span=55, adjust=False).mean()

# 布林带 (20, 2σ)
d["bb_mid"]   = d["close"].rolling(20).mean()
d["bb_std"]   = d["close"].rolling(20).std()
d["bb_upper"] = d["bb_mid"] + 2 * d["bb_std"]
d["bb_lower"] = d["bb_mid"] - 2 * d["bb_std"]

# RSI (14)
delta = d["close"].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
d["rsi"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))

# MACD (12, 26, 9)
ema12 = d["close"].ewm(span=12, adjust=False).mean()
ema26 = d["close"].ewm(span=26, adjust=False).mean()
d["macd"]        = ema12 - ema26
d["macd_signal"] = d["macd"].ewm(span=9, adjust=False).mean()
d["macd_hist"]   = d["macd"] - d["macd_signal"]

# 成交量均线
d["vol_ma5"]  = d["volume"].rolling(5).mean()
d["vol_ma20"] = d["volume"].rolling(20).mean()

d = d.dropna().reset_index(drop=True)
print(f"计算完成，有效行数: {len(d)}")

## 4. K 线图（含均线 + 布林带 + 成交量 + RSI + MACD）

In [ ]:
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.55, 0.15, 0.15, 0.15],
    vertical_spacing=0.02,
    subplot_titles=(f"{SYMBOL}  K线", "成交量", "RSI (14)", "MACD"),
)

t = d["timestamp"]

# ── Row 1: K 线 ──────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=t, open=d["open"], high=d["high"], low=d["low"], close=d["close"],
    name="K线",
    increasing_line_color="#ef5350",
    decreasing_line_color="#26a69a",
    increasing_fillcolor="#ef5350",
    decreasing_fillcolor="#26a69a",
), row=1, col=1)

for col, color, dash, name in [
    ("ema9",  "#ffeb3b", "solid", "EMA9"),
    ("ema21", "#ff9800", "solid", "EMA21"),
    ("ema55", "#ce93d8", "solid", "EMA55"),
]:
    fig.add_trace(go.Scatter(x=t, y=d[col], line=dict(color=color, width=1, dash=dash), name=name), row=1, col=1)

# 布林带
fig.add_trace(go.Scatter(
    x=t, y=d["bb_upper"], line=dict(color="rgba(100,181,246,0.6)", width=1, dash="dot"), name="BB上轨",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=t, y=d["bb_lower"], line=dict(color="rgba(100,181,246,0.6)", width=1, dash="dot"), name="BB下轨",
    fill="tonexty", fillcolor="rgba(100,181,246,0.05)",
), row=1, col=1)

# ── Row 2: 成交量 ────────────────────────────────────────────────
colors = ["#ef5350" if c >= o else "#26a69a" for c, o in zip(d["close"], d["open"])]
fig.add_trace(go.Bar(x=t, y=d["volume"], marker_color=colors, name="成交量", showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=d["vol_ma5"],  line=dict(color="#ffeb3b", width=1), name="VOL MA5"),  row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=d["vol_ma20"], line=dict(color="#ff9800", width=1), name="VOL MA20"), row=2, col=1)

# ── Row 3: RSI ───────────────────────────────────────────────────
fig.add_trace(go.Scatter(x=t, y=d["rsi"], line=dict(color="#ba68c8", width=1.5), name="RSI"), row=3, col=1)
fig.add_hline(y=70, line=dict(color="red",  width=1, dash="dash"), row=3, col=1)
fig.add_hline(y=30, line=dict(color="green", width=1, dash="dash"), row=3, col=1)
fig.add_hrect(y0=30, y1=70, fillcolor="rgba(255,255,255,0.03)", line_width=0, row=3, col=1)

# ── Row 4: MACD ──────────────────────────────────────────────────
hist_colors = ["#ef5350" if v >= 0 else "#26a69a" for v in d["macd_hist"]]
fig.add_trace(go.Bar(x=t, y=d["macd_hist"], marker_color=hist_colors, name="MACD Hist", showlegend=False), row=4, col=1)
fig.add_trace(go.Scatter(x=t, y=d["macd"],        line=dict(color="#2196f3", width=1.5), name="MACD"),   row=4, col=1)
fig.add_trace(go.Scatter(x=t, y=d["macd_signal"], line=dict(color="#ff9800", width=1.5), name="Signal"), row=4, col=1)

# ── 布局 ─────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(text=f"{SYMBOL}  {START_DATE} ~ {END_DATE}", font=dict(size=16)),
    height=900,
    template="plotly_dark",
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
    margin=dict(l=60, r=20, t=80, b=20),
    hovermode="x unified",
)
fig.update_yaxes(title_text="价格",    row=1, col=1)
fig.update_yaxes(title_text="成交量",  row=2, col=1)
fig.update_yaxes(title_text="RSI",    row=3, col=1, range=[0, 100])
fig.update_yaxes(title_text="MACD",   row=4, col=1)

fig.show()

## 5. 收益率分布

In [ ]:
ret = d["close"].pct_change().dropna()
log_ret = np.log(d["close"] / d["close"].shift(1)).dropna()

fig2 = make_subplots(rows=1, cols=2, subplot_titles=("简单收益率分布", "对数收益率分布"))

for col_idx, (r, name) in enumerate([(ret, "简单收益率"), (log_ret, "对数收益率")], start=1):
    fig2.add_trace(go.Histogram(
        x=r, nbinsx=80,
        marker_color="rgba(100,181,246,0.7)",
        name=name, showlegend=False,
    ), row=1, col=col_idx)
    # 零线
    fig2.add_vline(x=0, line=dict(color="red", dash="dash", width=1), row=1, col=col_idx)

ann = (
    f"均值: {ret.mean():.4f}<br>"
    f"标准差: {ret.std():.4f}<br>"
    f"偏度: {ret.skew():.4f}<br>"
    f"峰度: {ret.kurt():.4f}<br>"
    f"年化收益: {(1 + ret.mean()) ** 365 - 1:.2%}<br>"
    f"年化波动: {ret.std() * np.sqrt(365):.2%}<br>"
    f"年化夏普: {ret.mean() / ret.std() * np.sqrt(365):.2f}"
)
fig2.add_annotation(
    xref="paper", yref="paper", x=1.01, y=0.95,
    text=ann, showarrow=False, align="left",
    font=dict(size=12, color="white"),
    bordercolor="gray", borderwidth=1, bgcolor="rgba(0,0,0,0.5)",
)

fig2.update_layout(
    title=f"{SYMBOL} 收益率分布  {START_DATE} ~ {END_DATE}",
    height=420, template="plotly_dark",
    margin=dict(l=60, r=180, t=60, b=40),
)
fig2.show()

## 6. 关键统计摘要

In [ ]:
price_chg = (d["close"].iloc[-1] / d["close"].iloc[0] - 1) * 100
high_price = d["high"].max()
low_price  = d["low"].min()
drawdown   = (d["close"] / d["close"].cummax() - 1)
max_dd     = drawdown.min() * 100
avg_vol    = d["volume"].mean()

print(f"{'═' * 40}")
print(f"  {SYMBOL}  {START_DATE} ~ {END_DATE}")
print(f"{'═' * 40}")
print(f"  期间涨跌幅   : {price_chg:+.2f}%")
print(f"  最高价       : {high_price:,.2f}")
print(f"  最低价       : {low_price:,.2f}")
print(f"  最大回撤     : {max_dd:.2f}%")
print(f"  年化收益     : {(1 + ret.mean()) ** 365 - 1:.2%}")
print(f"  年化波动率   : {ret.std() * np.sqrt(365):.2%}")
print(f"  年化夏普比率 : {ret.mean() / ret.std() * np.sqrt(365):.2f}")
print(f"  平均成交量   : {avg_vol:,.0f}")
print(f"{'═' * 40}")